[← Lekcja 3: Przyspieszenie](https://colab.research.google.com/github/InteraktywnaFizykaZPytonem/Interaktywna-Fizyka-z-Pytonem/blob/main/Kinematyka_3_przyspieszenie.ipynb) &nbsp;·&nbsp; [📚 Spis lekcji](https://github.com/InteraktywnaFizykaZPytonem/Interaktywna-Fizyka-z-Pytonem#spis-lekcji) &nbsp;·&nbsp; **Lekcja 4 z 4** (następne wkrótce)

# 🐝 Kinematyka – część 4: Pszczoła krąży (ruch po okręgu, współrzędne walcowe i kuliste)

Do tej pory opisywaliśmy położenie trzema odległościami $x$, $y$, $z$. To dobre, gdy pszczoła leci prosto. Ale gdy **krąży** wokół latarni, jej współrzędne $x$ i $y$ zmieniają się cały czas, mimo że odległość od latarni jest stała. Dla takich ruchów wygodniejsze są **inne układy współrzędnych**. W tej lekcji:

1. poznasz **współrzędne walcowe** $(\rho;\ \varphi;\ z)$ i **kuliste** $(r;\ \theta;\ \varphi)$,
2. opiszesz **ruch po okręgu**: prędkość kątowa, okres, prędkość i przyspieszenie dośrodkowe,
3. zobaczysz **ruch po spirali** jako zwykły ruch we współrzędnych walcowych.

> 💾 Chcesz zachować swoje zmiany? Wybierz *Plik → Zapisz kopię na Dysku*.

> ⚠️ **Zobaczyłeś błąd `NameError: name '...' is not defined`?** To znaczy, że pominąłeś jakąś komórkę wyżej albo Colab zapomniał zmienne po przerwie. Uruchom komórki po kolei od góry albo wybierz *Środowisko wykonawcze → Uruchom wszystko*.

Komórki uruchamiaj **po kolei od góry**. Zacznij od kodu pomocniczego:

In [ ]:
#@title 🔧 Kod pomocniczy – rysowanie (uruchom i nie przejmuj się nim, wrócimy do niego później)
import numpy as np
import plotly.graph_objects as go

R_PSZCZOLY = 0.25   # promień pszczoły [m]

# kamera: widok domyślny i widoki z różnych stron
WIDOKI = {
    "↺ Reset widoku": dict(eye=dict(x=1.6, y=-1.6, z=1.0), up=dict(x=0, y=0, z=1)),
    "Z góry":        dict(eye=dict(x=0.0, y=0.001, z=2.6), up=dict(x=0, y=1, z=0)),
    "Z przodu":      dict(eye=dict(x=0.0, y=-2.6, z=0.0), up=dict(x=0, y=0, z=1)),
    "Z boku":        dict(eye=dict(x=2.6, y=0.0, z=0.0), up=dict(x=0, y=0, z=1)),
}

def pszczola(p, R=R_PSZCZOLY):
    """Kula w żółto-czarne paski ze środkiem w punkcie p."""
    x, y, z = p
    u, v = np.meshgrid(np.linspace(0, 2*np.pi, 40), np.linspace(0, np.pi, 40))
    return go.Surface(
        x=x + R*np.cos(u)*np.sin(v), y=y + R*np.sin(u)*np.sin(v), z=z + R*np.cos(v),
        surfacecolor=(np.floor(v/(np.pi/7)) % 2),
        colorscale=[[0, "#FFD400"], [0.5, "#FFD400"], [0.5, "#1a1a1a"], [1, "#1a1a1a"]],
        cmin=0, cmax=1, showscale=False, hoverinfo="skip",
        lighting=dict(ambient=0.9, diffuse=0.3, specular=0.05),
    )

def strzalka(poczatek, w, kolor, dl_grotu=0.6, r_grotu=0.18):
    """Strzałka-wektor: trzon (linia) + grot (stożek z trójkątów). Czubek leży w punkcie poczatek + w."""
    poczatek = np.asarray(poczatek, float)
    w = np.asarray(w, float)
    d = np.linalg.norm(w)
    e = w / d if d > 1e-9 else np.array([0, 0, 1.0])
    dl = min(dl_grotu, d)                          # krótkie wektory dostają mniejszy grot
    r = r_grotu * dl / dl_grotu
    ostrze = poczatek + w
    podstawa = ostrze - e * dl
    trzon = go.Scatter3d(x=[poczatek[0], podstawa[0]], y=[poczatek[1], podstawa[1]],
                         z=[poczatek[2], podstawa[2]], mode="lines",
                         line=dict(color=kolor, width=7), hoverinfo="skip")
    a = np.cross(e, [1, 0, 0]) if abs(e[0]) < 0.9 else np.cross(e, [0, 1, 0])
    a = a / np.linalg.norm(a)
    b = np.cross(e, a)
    t = np.linspace(0, 2*np.pi, 16, endpoint=False)
    kolo = podstawa + r*(np.outer(np.cos(t), a) + np.outer(np.sin(t), b))
    punkty = np.vstack([ostrze, kolo, podstawa])   # 0 = czubek, 1..16 = okrąg, 17 = środek podstawy
    n = len(t)
    boki = [(0, 1 + k, 1 + (k + 1) % n) for k in range(n)]
    spod = [(n + 1, 1 + (k + 1) % n, 1 + k) for k in range(n)]
    ij = np.array(boki + spod)
    grot = go.Mesh3d(x=punkty[:, 0], y=punkty[:, 1], z=punkty[:, 2],
                     i=ij[:, 0], j=ij[:, 1], k=ij[:, 2], color=kolor, flatshading=True,
                     lighting=dict(ambient=0.8, diffuse=0.4), hoverinfo="skip")
    return [trzon, grot]

def osie_ukladu(L):
    return go.Scatter3d(x=[-L, L, None, 0, 0, None, 0, 0], y=[0, 0, None, -L, L, None, 0, 0],
                        z=[0, 0, None, 0, 0, None, -L, L], mode="lines",
                        line=dict(color="gray", width=3), hoverinfo="skip")

def menu_widokow(y):
    return dict(type="buttons", direction="right", x=0.0, xanchor="left", y=y, yanchor="bottom",
                buttons=[dict(label=n, method="relayout", args=[{"scene.camera": k}])
                         for n, k in WIDOKI.items()])

def menu_przelacznikow(grupy, y, krok=0.25):
    """Dla każdej grupy dwa przyciski: pokaż / ukryj. Grupy działają niezależnie od siebie,
    więc można ukryć kilka elementów naraz.
    grupy = [(nazwa, [indeksy śladów], zmiana_układu_po_pokazaniu, zmiana_układu_po_ukryciu), ...]"""
    menu = []
    for k, (nazwa, indeksy, po_pokazaniu, po_ukryciu) in enumerate(grupy):
        menu.append(dict(
            type="buttons", direction="right", x=krok*k, xanchor="left", y=y, yanchor="bottom",
            showactive=True, active=0,
            buttons=[dict(label=f"{nazwa}: pokaż", method="update", args=[{"visible": True}, po_pokazaniu, indeksy]),
                     dict(label="ukryj", method="update", args=[{"visible": False}, po_ukryciu, indeksy])]))
    return menu

def zrob_uklad(tytul, zakres, dtick, adnotacje=(), grupy=(), dodatkowe_menu=(), sliders=(),
               gora=150, dol=0, wysokosc=720, krok_grup=0.25):
    def os_(nazwa):
        return dict(title=nazwa, range=[-zakres, zakres], dtick=dtick)
    wys_wykresu = wysokosc - gora - dol
    menu = [menu_widokow(1 + 40/wys_wykresu)] + menu_przelacznikow(grupy, 1.0, krok_grup) + list(dodatkowe_menu)
    return go.Layout(
        title=dict(text=tytul, x=0.5, y=0.99, yanchor="top"),
        scene=dict(xaxis=os_("x [m]"), yaxis=os_("y [m]"), zaxis=os_("z [m]"), aspectmode="cube",
                   camera=WIDOKI["↺ Reset widoku"], annotations=list(adnotacje)),
        height=wysokosc, margin=dict(l=0, r=0, t=gora, b=dol), showlegend=False,
        updatemenus=menu, sliders=list(sliders),
        uirevision="stala",   # zachowuje obrót wykresu podczas animacji i zmian
    )

ZOLTY = "#F2B600"       # kolor wektora przyspieszenia

def kontrolki_animacji(czasy, czas_klatki, redraw, odstep):
    """Suwak czasu oraz przyciski Start / Pauza."""
    suwak = dict(active=0, x=0.12, len=0.80, xanchor="left", y=0, yanchor="top",
                 pad=dict(t=odstep, b=10), font=dict(color="rgba(0,0,0,0)"),
                 currentvalue=dict(prefix="t = ", suffix=" s", font=dict(size=16)),
                 steps=[dict(method="animate", label=f"{t:.1f}",
                             args=[[str(k)], dict(mode="immediate",
                                                  frame=dict(duration=0, redraw=redraw),
                                                  transition=dict(duration=0))])
                        for k, t in enumerate(czasy)])
    odtwarzanie = dict(type="buttons", direction="left", x=0.0, xanchor="left", y=0, yanchor="top",
                       pad=dict(t=odstep), showactive=False, buttons=[
        dict(label="▶ Start", method="animate",
             args=[None, dict(frame=dict(duration=czas_klatki, redraw=redraw), fromcurrent=True,
                              transition=dict(duration=0))]),
        dict(label="⏸ Pauza", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate",
                                transition=dict(duration=0))])])
    return suwak, odtwarzanie

def strzalka_2d(p0, w, d_grotu=0.5, kat=25):
    """Współrzędne łamanej rysującej strzałkę 2D od p0 do p0 + w."""
    p0 = np.asarray(p0, float); w = np.asarray(w, float)
    d = np.linalg.norm(w)
    if d < 1e-9:
        return [p0[0]], [p0[1]]
    e = w / d
    ostrze = p0 + w
    dl = min(d_grotu, 0.35*d)
    th = np.radians(kat)
    obrot = lambda f: np.array([[np.cos(f), -np.sin(f)], [np.sin(f), np.cos(f)]])
    lewy = ostrze - dl*(obrot(th) @ e)
    prawy = ostrze - dl*(obrot(-th) @ e)
    return ([p0[0], ostrze[0], lewy[0], ostrze[0], prawy[0]],
            [p0[1], ostrze[1], lewy[1], ostrze[1], prawy[1]])

def zakres_osi(*tablice):
    lo = min(float(np.min(x)) for x in tablice)
    hi = max(float(np.max(x)) for x in tablice)
    margines = 0.12*(hi - lo) if hi > lo else 0.5
    return [lo - margines, hi + margines]



from plotly.subplots import make_subplots

L = 5   # osie 3D rysujemy od -L do L

def luk(srodek, a, b, promien, kat0, kat1, n=30):
    """Punkty łuku okręgu leżącego w płaszczyźnie rozpiętej przez wersory a i b."""
    s = np.linspace(kat0, kat1, n)
    srodek = np.asarray(srodek, float); a = np.asarray(a, float); b = np.asarray(b, float)
    return srodek + promien*(np.outer(np.cos(s), a) + np.outer(np.sin(s), b))

def linia3d(punkty, kolor, szerokosc=7, kreska="solid"):
    p = np.asarray(punkty, float)
    return go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode="lines",
                        line=dict(color=kolor, width=szerokosc, dash=kreska), hoverinfo="skip")

def etykieta(p, tekst, kolor):
    return dict(x=float(p[0]), y=float(p[1]), z=float(p[2]), text=f"<b>{tekst}</b>", showarrow=False,
                font=dict(color=kolor, size=22))

def powierzchnia(x, y, z):
    return go.Surface(x=x, y=y, z=z, surfacecolor=np.zeros_like(x), cmin=0, cmax=1, showscale=False,
                      colorscale=[[0, "#9bb7d4"], [1, "#9bb7d4"]], opacity=0.15, hoverinfo="skip")

# ---------------------------------------------------------------------
# CZĘŚĆ 1: współrzędne walcowe i kuliste (wykres z suwakami)
# ---------------------------------------------------------------------
def walcowe_slady(rho, phi_deg, z):
    """0 pszczoła | 1-2 wektor r | 3 ρ | 4 z | 5 kąt φ | 6 walec | 7 osie"""
    phi = np.radians(phi_deg)
    e = np.array([np.cos(phi), np.sin(phi), 0.0])
    P = np.array([rho*np.cos(phi), rho*np.sin(phi), z])
    Q = np.array([P[0], P[1], 0.0])                    # rzut pszczoły na płaszczyznę xy
    ra = min(max(0.4*rho, 0.6), 2.0)
    u, h = np.meshgrid(np.linspace(0, 2*np.pi, 40), np.linspace(-L, L, 2))
    slady = ([pszczola(P)] + strzalka([0, 0, 0], P, "red")
             + [linia3d([[0, 0, 0], Q], "darkorange"), linia3d([Q, P], "steelblue"),
                linia3d(luk([0, 0, 0], [1, 0, 0], [0, 1, 0], ra, 0, phi), "purple", 6),
                powierzchnia(rho*np.cos(u), rho*np.sin(u), h), osie_ukladu(L)])
    adn = {"wektor": [etykieta(P/2 + np.array([0, 0, 0.4]), "r", "red")],
           "pomocnicze": [etykieta(Q/2 + np.array([0, 0, -0.4]), "ρ", "darkorange"),
                          etykieta(1.4*ra*np.array([np.cos(phi/2), np.sin(phi/2), 0]), "φ", "purple"),
                          etykieta((P + Q)/2 + 0.4*e, "z", "steelblue")]}
    return slady, adn

def kuliste_slady(r, theta_deg, phi_deg):
    """0 pszczoła | 1-2 wektor r | 3 rzuty | 4 kąt θ | 5 kąt φ | 6 kula | 7 osie"""
    th = np.radians(theta_deg); phi = np.radians(phi_deg)
    e = np.array([np.cos(phi), np.sin(phi), 0.0])
    P = r*np.array([np.sin(th)*np.cos(phi), np.sin(th)*np.sin(phi), np.cos(th)])
    Q = np.array([P[0], P[1], 0.0])
    ra = min(max(0.4*r, 0.6), 2.0)
    rb = min(max(0.4*np.hypot(P[0], P[1]), 0.6), 2.0)
    u, v = np.meshgrid(np.linspace(0, 2*np.pi, 40), np.linspace(0, np.pi, 30))
    rzuty = go.Scatter3d(x=[0, Q[0], None, Q[0], P[0]], y=[0, Q[1], None, Q[1], P[1]],
                         z=[0, Q[2], None, Q[2], P[2]], mode="lines",
                         line=dict(color="gray", width=4, dash="dash"), hoverinfo="skip")
    slady = ([pszczola(P)] + strzalka([0, 0, 0], P, "red")
             + [rzuty, linia3d(luk([0, 0, 0], [0, 0, 1], e, ra, 0, th), "darkorange", 6),
                linia3d(luk([0, 0, 0], [1, 0, 0], [0, 1, 0], rb, 0, phi), "purple", 6),
                powierzchnia(r*np.cos(u)*np.sin(v), r*np.sin(u)*np.sin(v), r*np.cos(v)), osie_ukladu(L)])
    adn = {"wektor": [etykieta(P/2 + np.array([0, 0, 0.4]), "r", "red")],
           "pomocnicze": [etykieta(1.4*ra*(np.cos(th/2)*np.array([0, 0, 1.0]) + np.sin(th/2)*e), "θ", "darkorange"),
                          etykieta(1.4*rb*np.array([np.cos(phi/2), np.sin(phi/2), 0]), "φ", "purple")]}
    return slady, adn

def tytul_walcowe(rho, phi, z):
    p = np.radians(phi)
    return (f"walcowe (<b>ρ</b>; <b>φ</b>; <b>z</b>) = ({rho:g} m; {phi:g}°; {z:g} m)"
            f"<br><sup>kartezjańskie (x; y; z) = ({rho*np.cos(p):.2f}; {rho*np.sin(p):.2f}; {z:g}) m</sup>")

def tytul_kuliste(r, theta, phi):
    t, p = np.radians(theta), np.radians(phi)
    return (f"kuliste (<b>r</b>; <b>θ</b>; <b>φ</b>) = ({r:g} m; {theta:g}°; {phi:g}°)"
            f"<br><sup>kartezjańskie (x; y; z) = ({r*np.sin(t)*np.cos(p):.2f}; "
            f"{r*np.sin(t)*np.sin(p):.2f}; {r*np.cos(t):.2f}) m</sup>")

GRUPY_WALCOWE = [("pszczołę", [0], None), ("wektor r", [1, 2], "wektor"),
                 ("ρ, φ, z", [3, 4, 5], "pomocnicze"), ("walec ρ = const", [6], None)]
GRUPY_KULISTE = [("pszczołę", [0], None), ("wektor r", [1, 2], "wektor"),
                 ("θ, φ i rzuty", [3, 4, 5], "pomocnicze"), ("kulę r = const", [6], None)]

def suwaki_3d(zbuduj, tytul, specyfikacje, grupy):
    """Wykres 3D sterowany suwakami i polami wyboru (pokaż / ukryj elementy)."""
    from ipywidgets import FloatSlider, Checkbox, VBox, HBox
    from IPython.display import display
    try:                                   # nowe wersje plotly potrzebują pakietu anywidget
        import anywidget
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anywidget"])
    try:                                   # w Colabie wykres-widżet wymaga tego włączenia
        from google.colab import output
        output.enable_custom_widget_manager()
    except Exception:
        pass
    suwaki = [FloatSlider(min=lo, max=hi, step=krok, value=start, description=opis)
              for opis, lo, hi, krok, start in specyfikacje]
    pola = [Checkbox(value=True, description=f"Pokaż: {napis}") for napis, _, _ in grupy]
    wartosci = lambda: [s.value for s in suwaki]

    def etykiety(adn):
        return [a for (_, _, klucz), pole in zip(grupy, pola) if klucz and pole.value for a in adn[klucz]]

    slady, adn = zbuduj(*wartosci())
    wykres = go.FigureWidget(data=slady, layout=zrob_uklad(tytul(*wartosci()), L, 1, adnotacje=etykiety(adn)))

    def aktualizuj(zmiana=None):
        nowe, adn = zbuduj(*wartosci())
        with wykres.batch_update():
            for stary, nowy in zip(wykres.data, nowe):
                dane = nowy.to_plotly_json()
                dane.pop("type", None)
                stary.update(dane)
            for (_, indeksy, _), pole in zip(grupy, pola):
                for i in indeksy:
                    wykres.data[i].visible = pole.value
            wykres.layout.title.text = tytul(*wartosci())
            wykres.layout.scene.annotations = etykiety(adn)

    for w in suwaki + pola:
        w.observe(aktualizuj, names="value")
    display(VBox([VBox(suwaki), HBox(pola), wykres]))

def walcowe_z_suwakami(rho=3, phi=50, z=2):
    suwaki_3d(walcowe_slady, tytul_walcowe,
              [("ρ [m]", 0, 5, 0.5, rho), ("φ [°]", 0, 360, 1, phi), ("z [m]", -5, 5, 0.5, z)], GRUPY_WALCOWE)

def kuliste_z_suwakami(r=4, theta=50, phi=40):
    suwaki_3d(kuliste_slady, tytul_kuliste,
              [("r [m]", 0, 5, 0.5, r), ("θ [°]", 0, 180, 1, theta), ("φ [°]", 0, 360, 1, phi)], GRUPY_KULISTE)

# ---------------------------------------------------------------------
# CZĘŚĆ 2: ruch po okręgu (widok z góry + wykresy)
# ---------------------------------------------------------------------
def ruch_po_okregu(R, omega, czas, phi0=0):
    """Pszczoła krąży po okręgu o promieniu R z prędkością kątową omega [rad/s]. phi0 w stopniach."""
    if R <= 0 or czas <= 0:
        print("⚠️ Promień R i czas muszą być większe od zera.")
        return
    N = int(np.clip(24*abs(omega)*czas/(2*np.pi), 60, 240))
    t = np.linspace(0, czas, N + 1)
    phi = np.radians(phi0) + omega*t
    x, y = R*np.cos(phi), R*np.sin(phi)
    vx, vy = -omega*R*np.sin(phi), omega*R*np.cos(phi)
    ax, ay = -omega**2*R*np.cos(phi), -omega**2*R*np.sin(phi)
    v_war, a_war = abs(omega)*R, omega**2*R
    sv = 0.6*R/v_war if v_war > 1e-9 else 1.0       # skale: strzałka v ma 0,6 R, strzałka a ma 0,5 R
    sa = 0.5*R/a_war if a_war > 1e-9 else 1.0

    def slady(k, pelne):
        def S(xs, ys, **styl):
            return go.Scatter(x=xs, y=ys, **(styl if pelne else {}))
        poz = np.array([x[k], y[k]])
        rx, ry = strzalka_2d([0, 0], poz)
        vx_, vy_ = strzalka_2d(poz, sv*np.array([vx[k], vy[k]]))
        ax_, ay_ = strzalka_2d(poz, sa*np.array([ax[k], ay[k]]))
        kat = phi[k] - 2*np.pi*np.fix(phi[k]/(2*np.pi))      # kąt zredukowany do jednego obrotu
        s = np.linspace(0, kat, 30)
        ciche = dict(hoverinfo="skip", showlegend=False)
        return [
            S([x[k]], [y[k]], mode="markers",
              marker=dict(size=18, color="#FFD400", line=dict(color="black", width=3)), **ciche),     # 0 pszczoła
            S(rx, ry, mode="lines", line=dict(color="red", width=3), **ciche),                         # 1 r
            S(vx_, vy_, mode="lines", line=dict(color="royalblue", width=5), **ciche),                 # 2 v
            S(ax_, ay_, mode="lines", line=dict(color=ZOLTY, width=6), **ciche),                       # 3 a
            S(0.3*R*np.cos(s), 0.3*R*np.sin(s), mode="lines", line=dict(color="purple", width=4), **ciche),  # 4 kąt φ
            S(t[:k + 1], x[:k + 1], mode="lines", line=dict(color="red", width=3), name="x", showlegend=True),
            S(t[:k + 1], y[:k + 1], mode="lines", line=dict(color="red", width=3, dash="dash"), name="y", showlegend=True),
            S(t[:k + 1], vx[:k + 1], mode="lines", line=dict(color="royalblue", width=3), name="vx", showlegend=True),
            S(t[:k + 1], vy[:k + 1], mode="lines", line=dict(color="royalblue", width=3, dash="dash"), name="vy", showlegend=True),
            S(t[:k + 1], np.full(k + 1, v_war), mode="lines", line=dict(color="gray", width=3), name="|v|", showlegend=True),
            S(t[:k + 1], ax[:k + 1], mode="lines", line=dict(color=ZOLTY, width=3), name="ax", showlegend=True),
            S(t[:k + 1], ay[:k + 1], mode="lines", line=dict(color=ZOLTY, width=3, dash="dash"), name="ay", showlegend=True),
            S(t[:k + 1], np.full(k + 1, a_war), mode="lines", line=dict(color="gray", width=3, dash="dot"), name="|a|", showlegend=True),
        ]

    miejsca = [(1, 1)]*5 + [(2, 1)]*2 + [(2, 2)]*3 + [(2, 3)]*3
    fig = make_subplots(rows=2, cols=3, row_heights=[0.5, 0.5], vertical_spacing=0.13,
                        horizontal_spacing=0.07, specs=[[{"colspan": 3}, None, None], [{}, {}, {}]],
                        subplot_titles=("", "Położenie x(t), y(t)", "Prędkość vx(t), vy(t)", "Przyspieszenie ax(t), ay(t)"))
    for tr, (wiersz, kolumna) in zip(slady(0, True), miejsca):
        fig.add_trace(tr, row=wiersz, col=kolumna)
    s = np.linspace(0, 2*np.pi, 120)
    fig.add_trace(go.Scatter(x=R*np.cos(s), y=R*np.sin(s), mode="lines", hoverinfo="skip", showlegend=False,
                             line=dict(color="gray", width=2, dash="dot")), row=1, col=1)           # 13 okrąg
    fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers", hoverinfo="skip", showlegend=False,
                             marker=dict(size=7, color="black")), row=1, col=1)                     # 14 środek
    fig.frames = [go.Frame(data=slady(k, False), traces=list(range(13)), name=str(k)) for k in range(N + 1)]

    gran = 1.6*R
    fig.update_xaxes(title_text="x [m]", range=[-gran, gran], constrain="domain", row=1, col=1)
    fig.update_yaxes(title_text="y [m]", range=[-gran, gran], scaleanchor="x", scaleratio=1,
                     constrain="domain", row=1, col=1)
    opisy = {1: ("[m]", (x, y)), 2: ("[m/s]", (vx, vy, [v_war])), 3: ("[m/s²]", (ax, ay, [a_war]))}
    for kolumna, (napis, tablice) in opisy.items():
        fig.update_xaxes(range=[0, czas], title_text="t [s]", row=2, col=kolumna)
        fig.update_yaxes(title_text=napis, range=zakres_osi(*tablice), row=2, col=kolumna)

    suwak, odtwarzanie = kontrolki_animacji(t, int(1000*min(czas, 10)/N), False, 60)
    grupy = [("Wektor r", [1], {}, {}), ("Wektor v", [2], {}, {}), ("Wektor a", [3], {}, {}), ("Kąt φ", [4], {}, {})]
    T = 2*np.pi/abs(omega) if abs(omega) > 1e-12 else float("inf")
    tytul = (f"<b>Ruch po okręgu</b>: R = {R:g} m, ω = {omega:g} rad/s, T = {T:.2f} s"
             f"<br><sup><span style=\"color:red\"><b>czerwony</b></span> – r,   "
             f"<span style=\"color:royalblue\"><b>niebieski</b></span> – v (1 m/s = {sv:.2g} m),   "
             f"<span style=\"color:{ZOLTY}\"><b>żółty</b></span> – a (1 m/s² = {sa:.2g} m),   "
             "<span style=\"color:purple\"><b>fioletowy</b></span> – kąt φ</sup>")
    fig.update_layout(title=dict(text=tytul, x=0.5, y=0.99, yanchor="top"), height=1050,
                      margin=dict(l=70, r=130, t=130, b=170),
                      updatemenus=menu_przelacznikow(grupy, 1.0) + [odtwarzanie], sliders=[suwak],
                      legend=dict(x=1.01, y=0.5, xanchor="left"), uirevision="stala")
    if np.isfinite(T):
        print(f"ω = {omega:g} rad/s  →  okres T = 2π/ω = {T:.2f} s,  częstotliwość f = 1/T = {1/T:.3f} Hz")
    print(f"Wartość prędkości v = ωR = {v_war:.2f} m/s,   przyspieszenie dośrodkowe a = ω²R = {a_war:.2f} m/s²")
    fig.show()

# ---------------------------------------------------------------------
# CZĘŚĆ 3: ruch po spirali (współrzędne walcowe w ruchu)
# ---------------------------------------------------------------------
def ruch_srubowy(R, omega, vz, czas, z0=0):
    """ρ = R, φ = ω t, z = z0 + vz t  (ruch śrubowy, czyli spirala)."""
    if R <= 0 or czas <= 0:
        print("⚠️ Promień R i czas muszą być większe od zera.")
        return
    N = int(np.clip(24*abs(omega)*czas/(2*np.pi), 60, 200))
    czasy = np.linspace(0, czas, N + 1)
    polozenie = lambda t: np.array([R*np.cos(omega*t), R*np.sin(omega*t), z0 + vz*t])
    predkosc = lambda t: np.array([-omega*R*np.sin(omega*t), omega*R*np.cos(omega*t), vz])
    przysp = lambda t: np.array([-omega**2*R*np.cos(omega*t), -omega**2*R*np.sin(omega*t), 0.0])
    tor = np.array([polozenie(t) for t in np.linspace(0, czas, 300)])
    v_war = float(np.hypot(omega*R, vz)); a_war = omega**2*R
    sv = 1.0 if v_war <= 3 else 3/v_war
    sa = 2/a_war if a_war > 1e-9 else 1.0
    zakres = int(np.ceil(max(6, np.abs(tor).max() + 1)))
    zakres += zakres % 2
    wersor_x, wersor_y = np.array([1.0, 0, 0]), np.array([0, 1.0, 0])

    def ruchome(t):   # 0 pszczoła | 1-2 r | 3-4 v | 5-6 a | 7 ρ | 8 z | 9 kąt φ
        p = polozenie(t); q = np.array([p[0], p[1], 0.0])
        faza = omega*t
        return ([pszczola(p)] + strzalka([0, 0, 0], p, "red")
                + strzalka(p, sv*predkosc(t), "royalblue") + strzalka(p, sa*przysp(t), ZOLTY)
                + [linia3d([[0, 0, 0], q], "darkorange", 6), linia3d([q, p], "steelblue", 6),
                   linia3d(luk([0, 0, 0], wersor_x, wersor_y, 0.4*R, 0, faza - 2*np.pi*np.fix(faza/(2*np.pi))),
                           "purple", 6)])

    u, h = np.meshgrid(np.linspace(0, 2*np.pi, 40), np.linspace(-zakres, zakres, 2))
    stale = [linia3d(tor, "gray", 3, "dash"),                                   # 10 tor
             powierzchnia(R*np.cos(u), R*np.sin(u), h),                          # 11 walec
             osie_ukladu(zakres)]                                                # 12 osie
    dane = ruchome(0) + stale
    ramki = [go.Frame(data=ruchome(t), traces=list(range(10)), name=str(k)) for k, t in enumerate(czasy)]
    suwak, odtwarzanie = kontrolki_animacji(czasy, int(1000*min(czas, 10)/N), True, 10)
    grupy = [("Pszczoła", [0], {}, {}), ("Wektor r", [1, 2], {}, {}), ("Wektor v", [3, 4], {}, {}),
             ("Wektor a", [5, 6], {}, {}), ("Walcowe", [7, 8, 9, 11], {}, {})]
    tytul = (f"<b>Ruch śrubowy</b>: ρ = {R:g} m,  φ = {omega:g}·t,  z = {z0:g} + {vz:g}·t"
             "<br><sup><span style=\"color:red\"><b>r</b></span>,   "
             f"<span style=\"color:royalblue\"><b>v</b></span> (1 m/s = {sv:.2g} m),   "
             f"<span style=\"color:{ZOLTY}\"><b>a</b></span> (1 m/s² = {sa:.2g} m);   "
             "<span style=\"color:darkorange\"><b>ρ</b></span>, "
             "<span style=\"color:steelblue\"><b>z</b></span>, "
             "<span style=\"color:purple\"><b>φ</b></span> – współrzędne walcowe</sup>")
    uklad = zrob_uklad(tytul, zakres, 2, grupy=grupy, dodatkowe_menu=[odtwarzanie], sliders=[suwak],
                       gora=150, dol=110, krok_grup=0.2)
    if abs(omega) > 1e-12:
        T = 2*np.pi/abs(omega)
        print(f"Okres obiegu T = {T:.2f} s,  skok spirali (wzrost z na jeden obrót) h = vz·T = {vz*T:.2f} m")
    print(f"Wartość prędkości |v| = {v_war:.2f} m/s,   przyspieszenie dośrodkowe a = ω²R = {a_war:.2f} m/s²")
    go.Figure(data=dane, layout=uklad, frames=ramki).show()

## 1. Kąty w radianach

Układy, które poznamy, używają **kątów**. W fizyce mierzymy je w **radianach**. Kąt $\varphi$ w radianach to stosunek długości łuku $s$ do promienia $R$:

$$\varphi = \frac{s}{R}$$

Pełny obrót (360°) to łuk o długości $2\pi R$, czyli kąt $2\pi \approx 6{,}28$ rad. Stąd: $180° = \pi$ rad, $90° = \pi/2$ rad.

Funkcje `np.sin`, `np.cos` w Pythonie **przyjmują radiany**. Do zamiany służą `np.radians` i `np.degrees`, a liczba $\pi$ to `np.pi`.

In [ ]:
kat_stopnie = 90
kat_radiany = np.radians(kat_stopnie)

print(f"{kat_stopnie}° = {kat_radiany:.4f} rad")
print(f"π/3 rad = {np.degrees(np.pi / 3):.1f}°")
print("Pełny obrót =", 2 * np.pi, "rad")
print("cos(60°) =", np.cos(np.radians(60)))

> ✏️ **Zadanie.** Zamień w głowie 45° i 270° na radiany (jako ułamki $\pi$), a potem sprawdź w Pythonie. Co się stanie, jeśli do `np.cos` wstawisz 60 *bez* zamiany na radiany?

## 2. Współrzędne walcowe $(\rho;\ \varphi;\ z)$

Wyobraź sobie, że pszczoła krąży wokół pionowego słupa (osi $z$). Wygodnie jest opisać jej położenie tak:

- $\rho$ – odległość od **osi $z$** (nie od początku układu!),
- $\varphi$ – **kąt** między osią $x$ a rzutem wektora położenia na płaszczyznę $xy$ (liczony od osi $x$, przeciwnie do ruchu wskazówek zegara),
- $z$ – wysokość, tak jak dotąd.

Nazwa pochodzi stąd, że punkty o stałym $\rho$ leżą na **powierzchni walca**.

**Przejście do współrzędnych kartezjańskich:**

$$x = \rho\cos\varphi, \qquad y = \rho\sin\varphi, \qquad z = z$$

**W drugą stronę:**

$$\rho = \sqrt{x^2 + y^2}, \qquad \varphi = \operatorname{atan2}(y,\ x), \qquad z = z$$

Funkcja $\operatorname{atan2}(y, x)$ (w Pythonie `np.arctan2(y, x)`, **najpierw $y$**) sama wybiera właściwą ćwiartkę. Zwykły arcus tangens by tego nie zrobił.

### Zobacz to na wykresie
Przesuwaj suwaki. W podtytule wykresu zobaczysz odpowiadające im współrzędne kartezjańskie. Pola wyboru pozwalają ukryć pszczołę, wektor $\vec{r}$, oznaczenia $\rho$, $\varphi$, $z$ lub walec. Wykres możesz obracać.

In [ ]:
walcowe_z_suwakami()

> ⚠️ Jeśli suwaki się pojawiły, ale wykres nie reaguje – uruchom komórkę ponownie.

> ✏️ **Obserwacje.** (1) Zmieniaj tylko $\varphi$ od 0° do 360°. Po jakiej krzywej porusza się pszczoła? Co się dzieje z $\rho$ i $z$? (2) Ustaw $\rho = 0$. Czy zmiana $\varphi$ ma wtedy jeszcze jakikolwiek wpływ na położenie? (3) Zmieniaj tylko $z$. W którą stronę porusza się pszczoła?

### Twój kod: zamiana współrzędnych
Pszczoła jest w punkcie kartezjańskim $(1{,}5;\ 2;\ 2)$ m. Policzmy jej współrzędne walcowe:

In [ ]:
x, y, z = 1.5, 2, 2

rho = np.sqrt(x**2 + y**2)
phi = np.arctan2(y, x)            # wynik w radianach

print(f"ρ = {rho:.2f} m")
print(f"φ = {phi:.3f} rad = {np.degrees(phi):.1f}°")
print(f"z = {z} m")

# sprawdzenie: z powrotem do współrzędnych kartezjańskich
print("x =", rho * np.cos(phi), "  y =", rho * np.sin(phi))

Teraz wpisz wyniki do wykresu (suwaki ustawią się w podanym punkcie) i sprawdź, czy podtytuł pokazuje $(1{,}50;\ 2{,}00;\ 2)$:

In [ ]:
walcowe_z_suwakami(rho=2.5, phi=53, z=2)

## 3. Współrzędne kuliste $(r;\ \theta;\ \varphi)$

Gdy ruch odbywa się wokół **punktu** (np. pszczoła krąży wokół kwiatka w trzech wymiarach), wygodne są współrzędne kuliste:

- $r$ – odległość od **początku układu** (długość wektora położenia, ta sama co w części 1!),
- $\theta$ – kąt między osią $z$ a wektorem $\vec{r}$ (od 0° do 180°),
- $\varphi$ – kąt azymutalny, **taki sam jak we współrzędnych walcowych**.

Punkty o stałym $r$ leżą na **powierzchni kuli**.

**Przejście do współrzędnych kartezjańskich:**

$$x = r\sin\theta\cos\varphi, \qquad y = r\sin\theta\sin\varphi, \qquad z = r\cos\theta$$

**W drugą stronę:**

$$r = \sqrt{x^2 + y^2 + z^2}, \qquad \theta = \arccos\frac{z}{r}, \qquad \varphi = \operatorname{atan2}(y,\ x)$$

Zauważ związek z poprzednim układem: odległość od osi $z$ to $\rho = r\sin\theta$.

> ℹ️ **Uwaga o oznaczeniach.** W niektórych podręcznikach matematyki litery $\theta$ i $\varphi$ są zamienione. My używamy **konwencji fizycznej** (zgodnej z normą ISO): $\theta$ to kąt od osi $z$.

> 🌍 **Zastosowanie:** współrzędne geograficzne to współrzędne kuliste Ziemi. Długość geograficzna to $\varphi$, a szerokość geograficzna to $90° - \theta$ (bo $\theta$ liczymy od bieguna północnego, a szerokość od równika).

### Zobacz to na wykresie

In [ ]:
kuliste_z_suwakami()

> ✏️ **Obserwacje.** (1) Zmieniaj tylko $\theta$ od 0° do 180°. Po jakiej krzywej porusza się pszczoła? Gdzie jest dla $\theta = 0°$, a gdzie dla $\theta = 90°$ i $180°$? (2) Zmieniaj tylko $\varphi$. Jak wygląda tor? (3) Zmieniaj tylko $r$. Czym jest tor pszczoły?

### Twój kod: ten sam punkt w współrzędnych kulistych

In [ ]:
x, y, z = 1.5, 2, 2

r = np.sqrt(x**2 + y**2 + z**2)
theta = np.arccos(z / r)
phi = np.arctan2(y, x)

print(f"r = {r:.2f} m")
print(f"θ = {np.degrees(theta):.1f}°")
print(f"φ = {np.degrees(phi):.1f}°")

Sprawdź na wykresie (podtytuł powinien pokazać $(1{,}50;\ 2{,}00;\ 2{,}00)$):

In [ ]:
kuliste_z_suwakami(r=3.2, theta=51, phi=53)

> ✏️ **Twój kod: Kraków na globusie.** Kraków leży na szerokości $50{,}06°$ N i długości $19{,}94°$ E. Promień Ziemi to $6371$ km. **Napisz kod**, który obliczy kartezjańskie współrzędne Krakowa w układzie ze środkiem w środku Ziemi i osią $z$ przez biegun północny.

In [ ]:
R_ziemi = 6371        # km
szerokosc = 50.06     # ° N
dlugosc = 19.94       # ° E

# Twój kod tutaj:
# 1) kąt theta = 90° - szerokość (zamień na radiany: np.radians)
# 2) wzory na x, y, z we współrzędnych kulistych

<details>
<summary>💡 Rozwiązanie</summary>

```python
theta = np.radians(90 - szerokosc)
phi = np.radians(dlugosc)

x = R_ziemi * np.sin(theta) * np.cos(phi)
y = R_ziemi * np.sin(theta) * np.sin(phi)
z = R_ziemi * np.cos(theta)
print(x, y, z)
```

Wynik: ok. $(3845;\ 1395;\ 4885)$ km. Dodatnie $z$ oznacza półkulę północną. Sprawdź: $\sqrt{x^2 + y^2 + z^2}$ powinno dać 6371 km.
</details>

## 4. Ruch po okręgu

Pszczoła krąży po okręgu o promieniu $R$ wokół osi $z$, ze **stałą prędkością kątową** $\omega$. We współrzędnych walcowych to bardzo proste: $\rho = R$ i $z = \text{const}$ są stałe, a zmienia się tylko kąt:

$$\varphi(t) = \varphi_0 + \omega\,t$$

**Prędkość kątowa** $\omega$ mówi, jaki kąt pszczoła zakreśla w jednostce czasu:

$$\omega = \frac{\Delta\varphi}{\Delta t} \quad [\text{rad/s}]$$

Czas jednego okrążenia to **okres** $T$, a liczba okrążeń na sekundę to **częstotliwość** $f$:

$$T = \frac{2\pi}{\omega}, \qquad f = \frac{1}{T} \quad [\text{Hz}]$$

**Wektor położenia** we współrzędnych kartezjańskich (to zwykłe wzory na przejście z walcowych):

$$x = R\cos(\varphi_0 + \omega t), \qquad y = R\sin(\varphi_0 + \omega t)$$

### Prędkość
Wzór $\vec{v} = \Delta\vec{r}/\Delta t$ dla bardzo krótkiego $\Delta t$ daje przemieszczenie wzdłuż **stycznej** do okręgu. W czasie $\Delta t$ pszczoła przelatuje łuk o długości $R\,\Delta\varphi = R\omega\,\Delta t$, więc:

$$|\vec{v}| = \omega R = \frac{2\pi R}{T}$$

Wektor $\vec{v}$ jest **prostopadły do $\vec{r}$**, a jego składowe to $v_x = -\omega R\sin\varphi$ i $v_y = \omega R\cos\varphi$.

### Przyspieszenie
Wartość prędkości jest stała, ale jej **kierunek** się zmienia, a więc według lekcji 3 pszczoła ma przyspieszenie! W czasie $\Delta t$ wektor $\vec{v}$ obraca się o kąt $\omega\,\Delta t$, więc zmienia się o $|\Delta\vec{v}| \approx v\,\omega\,\Delta t$. Stąd $a = v\omega$:

$$a = \omega^2 R = \frac{v^2}{R}, \qquad \vec{a} = -\omega^2\,\vec{r}$$

Przyspieszenie jest skierowane **do środka okręgu** (stąd nazwa: *dośrodkowe*), czyli prostopadle do prędkości. To dokładnie przypadek „skręcania" z lekcji 3.

### Sprawdźmy to numerycznie
Zamiast wierzyć wzorom, policzmy $\Delta\vec{r}/\Delta t$ i $\Delta\vec{v}/\Delta t$ dla bardzo małego $\Delta t$ i porównajmy z wzorami:

In [ ]:
R = 3
omega = 2
t = 1.0
dt = 0.0001        # bardzo krótki odcinek czasu

def polozenie(t):
    return np.array([R * np.cos(omega * t), R * np.sin(omega * t)])

def predkosc(t):
    return omega * R * np.array([-np.sin(omega * t), np.cos(omega * t)])

v_z_definicji = (polozenie(t + dt) - polozenie(t)) / dt     # Δr / Δt
a_z_definicji = (predkosc(t + dt) - predkosc(t)) / dt       # Δv / Δt

print("v z definicji Δr/Δt:", np.round(v_z_definicji, 3))
print("v ze wzoru:         ", np.round(predkosc(t), 3))
print("a z definicji Δv/Δt:", np.round(a_z_definicji, 3))
print("a ze wzoru -ω² r:   ", np.round(-omega**2 * polozenie(t), 3))

> 💡 **Uwaga o Pythonie:** słowo `def` **definiuje funkcję**: gotowy przepis, który bierze liczbę (tu czas `t`) i zwraca wynik (`return`). Dzięki temu nie musimy pisać tych samych wzorów po kilka razy.

### Zobacz animację
Pszczoła krąży po okręgu. Pod widokiem z góry powstają wykresy składowych położenia, prędkości i przyspieszenia. Fioletowy łuk to kąt $\varphi$. Przyciski *pokaż / ukryj* pozwalają ukrywać wektory (także kilka naraz).

In [ ]:
R = 3             # promień okręgu [m]
omega = 1.5       # prędkość kątowa [rad/s]
okrazenia = 2     # ile okrążeń pokazać

czas = okrazenia * 2 * np.pi / omega

ruch_po_okregu(R, omega, czas)

### Jak czytać wykresy?
- **Strzałki:** $\vec{v}$ (niebieska) jest zawsze **styczna** do okręgu i prostopadła do $\vec{r}$ (czerwonej). $\vec{a}$ (żółta) jest skierowana **do środka**, przeciwnie do $\vec{r}$.
- **Położenie:** $x(t)$ i $y(t)$ to fale (cosinus i sinus) **przesunięte względem siebie** o ćwierć okresu.
- **Prędkość i przyspieszenie:** składowe też są falami, a ich **wartości** $|\vec{v}|$ i $|\vec{a}|$ (szare linie) są **stałe**.
- **Związek:** $v_x$ jest „opóźnione o ćwierć okresu" względem $x$, a $a_x$ ma przeciwny znak niż $x$.

> ✏️ **Eksperymenty.** Zmieniaj parametry i **przewiduj** wynik:
> 1. Podwój $\omega$ (`omega = 3`). Co stanie się z $v$ i $a$? (Wzory: $v = \omega R$, $a = \omega^2 R$.)
> 2. Wróć do `omega = 1.5` i podwój promień (`R = 6`). Co stanie się z $v$ i $a$?
> 3. Ustaw `omega = -1.5`. W którą stronę krąży pszczoła? Co stało się z kierunkiem $\vec{v}$, a co z $\vec{a}$?
> 4. Dodaj początkowy kąt: `ruch_po_okregu(R, omega, czas, phi0=90)`. Gdzie startuje pszczoła?

> 🤔 **Pytanie.** Pszczoła leci ze stałą szybkością, a mimo to ma przyspieszenie. Dlaczego? Czy to się zgadza z definicją przyspieszenia z lekcji 3?

## 5. Ruch po spirali

Połączmy kółko ze wznoszeniem: pszczoła krąży wokół osi $z$ **i jednocześnie** leci w górę ze stałą prędkością $v_z$. We współrzędnych walcowych opis jest banalny:

$$\rho = R, \qquad \varphi = \omega t, \qquad z = z_0 + v_z\,t$$

Taki ruch to **ruch śrubowy** (spirala). Na wykresie pojawia się też niebieski walec, na którym leży tor, oraz oznaczenia współrzędnych walcowych: $\rho$ (pomarańczowy odcinek), $z$ (niebieski odcinek) i $\varphi$ (fioletowy łuk).

In [ ]:
R = 2         # promień [m]
omega = 2     # prędkość kątowa [rad/s]
vz = 0.5      # prędkość wznoszenia [m/s]
czas = 6      # czas lotu [s]

ruch_srubowy(R, omega, vz, czas)

> ✏️ **Obserwacje.** (1) Czy wektor $\vec{v}$ jest teraz prostopadły do $\vec{r}$? Spróbuj ukryć wektor $\vec{r}$ i patrz tylko na $\vec{v}$ i $\vec{a}$. (2) Przyspieszenie $\vec{a}$ nadal wskazuje do środka okręgu. Dlaczego nie ma składowej pionowej? (Podpowiedź: $v_z$ jest stałe.) (3) Ustaw `vz = -0.5`. Co się zmieni? (4) **Skok spirali** to wysokość, o którą pszczoła wznosi się podczas jednego okrążenia: $h = v_z T$. Jaki jest skok dla danych z komórki?

## 6. 🧪 Zadania z kodem

Napisz własny kod. Rozwiązania są w rozwijanych blokach.

### ✏️ Zadanie A: okres i przyspieszenie
Pszczoła krąży po okręgu o promieniu $R = 3$ m z okresem $T = 4$ s. **Oblicz** $\omega$, $f$, $v$ i $a$, a potem sprawdź wynik na animacji.

In [ ]:
R = 3
T = 4

# Twój kod tutaj:

<details>
<summary>💡 Rozwiązanie</summary>

```python
omega = 2 * np.pi / T
f = 1 / T
v = omega * R
a = omega**2 * R
print(omega, f, v, a)

ruch_po_okregu(R, omega, 2 * T)
```

Wyniki: $\omega \approx 1{,}57$ rad/s, $f = 0{,}25$ Hz, $v \approx 4{,}71$ m/s, $a \approx 7{,}40$ m/s².
</details>

### 🚀 Zadanie B (wyzwanie): limit przyspieszenia
Pszczoła wytrzyma przyspieszenie dośrodkowe najwyżej **10 m/s²**. Krąży po okręgu o promieniu $R = 2$ m. Jaka jest **największa możliwa** prędkość kątowa, prędkość liniowa i najkrótszy okres? Co się zmieni, gdy promień będzie mniejszy (np. $R = 0{,}5$ m): czy pszczoła może wtedy lecieć szybciej?

In [ ]:
a_max = 10     # największe wytrzymywane przyspieszenie [m/s²]
R = 2

# Twój kod tutaj:
# Podpowiedź: a = ω² R, więc ω = √(a / R)

<details>
<summary>💡 Rozwiązanie</summary>

```python
omega_max = np.sqrt(a_max / R)
v_max = omega_max * R
T_min = 2 * np.pi / omega_max
print(omega_max, v_max, T_min)

ruch_po_okregu(R, omega_max, 2 * T_min)
```

Wyniki: $\omega_{max} \approx 2{,}24$ rad/s, $v_{max} \approx 4{,}47$ m/s, $T_{min} \approx 2{,}81$ s. Ogólnie $v_{max} = \sqrt{a_{max} R}$: **mniejszy promień oznacza mniejszą dopuszczalną prędkość liniową** (dla $R = 0{,}5$ m jest to ok. 2,24 m/s), choć prędkość kątowa może być większa.
</details>

### 🚀 Zadanie C: pszczoła na spirali
Pszczoła ma wznieść się z $z = 0$ do $z = 6$ m, wykonując **dwa okrążenia** po okręgu o promieniu $R = 2$ m z okresem $T = 3$ s. **Oblicz** $\omega$, czas lotu, $v_z$, skok spirali i wartość prędkości $|\vec{v}| = \sqrt{(\omega R)^2 + v_z^2}$. Potem narysuj lot.

In [ ]:
R = 2
T = 3
wysokosc = 6
okrazenia = 2

# Twój kod tutaj:

<details>
<summary>💡 Rozwiązanie</summary>

```python
omega = 2 * np.pi / T
czas = okrazenia * T
vz = wysokosc / czas
skok = vz * T
v = np.sqrt((omega * R)**2 + vz**2)
print(omega, czas, vz, skok, v)

ruch_srubowy(R, omega, vz, czas)
```

Wyniki: $\omega \approx 2{,}09$ rad/s, czas lotu 6 s, $v_z = 1$ m/s, skok $h = 3$ m, $|\vec{v}| \approx 4{,}31$ m/s.
</details>

## 7. Podsumowanie

**Współrzędne walcowe** $(\rho;\ \varphi;\ z)$

$$x = \rho\cos\varphi, \qquad y = \rho\sin\varphi, \qquad \rho = \sqrt{x^2 + y^2}, \qquad \varphi = \operatorname{atan2}(y,\ x)$$

**Współrzędne kuliste** $(r;\ \theta;\ \varphi)$

$$x = r\sin\theta\cos\varphi, \qquad y = r\sin\theta\sin\varphi, \qquad z = r\cos\theta$$

$$r = \sqrt{x^2 + y^2 + z^2}, \qquad \theta = \arccos\frac{z}{r}$$

**Ruch po okręgu** (promień $R$, prędkość kątowa $\omega$)

$$\varphi(t) = \varphi_0 + \omega t, \qquad T = \frac{2\pi}{\omega}, \qquad f = \frac{1}{T}$$

**Prędkość** (styczna do okręgu, prostopadła do $\vec{r}$)

$$|\vec{v}| = \omega R = \frac{2\pi R}{T}$$

**Przyspieszenie dośrodkowe** (do środka okręgu)

$$a = \omega^2 R = \frac{v^2}{R}, \qquad \vec{a} = -\omega^2\,\vec{r}$$

**W Pythonie:** `np.radians`, `np.degrees`, `np.pi`, `np.arctan2(y, x)`, `np.arccos`, `def nazwa(t): return ...`

**Najważniejsze:**
- układ współrzędnych wybieramy tak, żeby opis ruchu był jak najprostszy: dla okręgu walcowe, dla ruchu wokół punktu kuliste,
- w ruchu po okręgu ze stałą szybkością **przyspieszenie nie jest zerowe**, bo zmienia się kierunek prędkości,
- ruch po spirali to „okrąg + wznoszenie": we współrzędnych walcowych trzy proste wzory.

---

[← Lekcja 3: Przyspieszenie](https://colab.research.google.com/github/InteraktywnaFizykaZPytonem/Interaktywna-Fizyka-z-Pytonem/blob/main/Kinematyka_3_przyspieszenie.ipynb) &nbsp;·&nbsp; [📚 Spis lekcji](https://github.com/InteraktywnaFizykaZPytonem/Interaktywna-Fizyka-z-Pytonem#spis-lekcji) &nbsp;·&nbsp; **Lekcja 4 z 4** (następne wkrótce)